### Create Delta tables

#### Function to create an external delta table

In [0]:
def createDeltaTable(db_name, table_name, file_type, path):
    """Pass the db_name, table_name, file format and path to this function to create a delta table"""
    spark.sql(f"""CREATE OR REPLACE TABLE {db_name}.{table_name} USING {file_type} LOCATION '{path}'""")

In [0]:
help(createDeltaTable)

#### Function to create an internal delta table

In [0]:
def createDeltaTableInternal(db_name,table_name):
    spark.sql(f"""CREATE DATABASE {db_name}""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {db_name}.{table_name} USING DELTA""")

In [0]:
createDeltaTableInternal('demo1','demo_table')

In [0]:
%sql
USE demo1;
SHOW TABLES;

#### Function to check if a path has a delta table

In [0]:
def DeltaCheck(path):    
    """This function takes path as argument and check if delta table exits at this path or not"""
    return DeltaTable.isDeltaTable(spark, path)

In [0]:
help(DeltaCheck)

### Operations in Delta tables

#### Merge statement
Using MERGE statement 3 standard manipulation language operation as follows can be performed in single statement
1. INSERT 
2. UPDATE
3. DELETE

When to use Delta Lake merge statement
1. to maintain SCD 
2. Change data capture: apply change sets from other data sources
3. INSERT, UPDATE OR DELETE data with dynamic matching conditions
4. Maintenance of VIEWS (Delta view)
5. GDPR compliance
6. when you want to apply selective changes to a delta table without rewriting the entire table.

In [0]:
%sql
MERGE INTO table_target AS t -- this will the historical (exiting) dataset
USING table_source AS s -- this will be incoming dataset
ON t.id = s.id
WHEN MATCHED THEN
  UPDATE SET
  t.id = s.id 
  t.firstname = s.firstname, -- updating first name
  t.last_name = s.last_name, -- updating last name
  t.gender = s.gender, -- updating gender
  t.salary = s.salary
WHEN NOT MATCHED 
  THEN INSERT (
    id, 
    first_name,
    last_name,
    gender,
    salary,
  )
  VALUES (
    s.id,
    s.first_name,
    s.last_anme,
    s.gender,
    s.salary
  )


#### History of delta table 

In [0]:
%sql
DESCRIBE HISTORY "/user/delta_table"

In [0]:
from delta import DeltaTable
DeltaTable.forPath(spark, "/user/delta_table/").history().display()

#### Read specific version of table

https://docs.delta.io/latest/delta-utility.html

In [0]:
%sql
DESCRIBE VERSION 

In [0]:
spark.read.format("delta").option("versionOf", 0).load(path1).display()

#### Restore delta table

In [0]:
df = spark.range(0,3)
df.display()
df.write.format("delta").option('mode', 'overwrite').save("/temp/df_delta")

In [0]:
display(dbutils.fs.ls("/temp/df_delta"))

In [0]:
df2 = spark.range(4,6)
df2.display()
df2.write.mode("overwrite").format("delta").save("/temp/df_delta")

In [0]:
df3 = spark.range(7,10)
df3.display()
df3.write.mode("overwrite").format("delta").save("/temp/df_delta")

In [0]:
display(dbutils.fs.ls("/temp/df_delta"))

In [0]:
# delta lake contains the latest version of delta table
spark.read.format("delta").load("temp/df_delta").show()

In [0]:
# show history of table
from delta import DeltaTable
DeltaTable.forPath(spark, "temp/df_delta").history().display()

In [0]:
# reading version 1 of delta table
spark.read.format("delta").option("versionAsOf", "1").load("temp/df_delta").display()

In [0]:
# restore to version 1 
from delta.tables import DeltaTable
DeltaTable.forPath(spark, "/temp/df_delta").restoreToVersion(1)

In [0]:
# reading the current version of delta table after restored to version 1 
spark.read.format("delta").load("temp/df_delta").display()

In [0]:
from delta.tables import DeltaTable
DeltaTable.forPath(spark, "temp/df_delta").history().display()

In [0]:
# before vacuum
spark.read.format("delta").option("versionAsOf", 0).load("/temp/df_delta").display()

In [0]:
# to set retentionDuration to false
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("myApp") \
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false") \
    .getOrCreate()

In [0]:
# to completely remove all previous versions of data from transaction log, use vacuum
DeltaTable.forPath(spark, "temp/df_delta").vacuum(retentionHours=0)

In [0]:
spark.read.format("delta").load("/temp/df_delta").display()

In [0]:
display(dbutils.fs.ls("/temp/df_delta"))

In [0]:
# after vacuum
spark.read.format("delta").option("versionAsOf", 0).load("/temp/df_delta").display()

In [0]:
%sql
CLEAR CACHE

#### Vaccum in Delta tables

### Generated Columns in Delta

In [0]:
 %sql
CREATE TABLE default.people10m (
  id INT,
  firstName STRING,
  middleName STRING,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  dateOfBirth DATE GENERATED ALWAYS AS (CAST(birthDate AS DATE)),
  ssn STRING,
  salary INT
)

### Drop a delta table
1. Internal delta table
      Steps :-
      1. DROP TABLE {delta table name}
2. External delta table

      Steps :- 
      1. DROP TABLE {delta table name}
      2. Go to ADLS location -> delete all the files (part00000.snappy.parquet + delta_log_) from the location where it has been created.<br>
          **Dropping a deltatable which is created at an external location will not remove the files from there**

### mergeSchema
df.write.format("type").option("inferSchema", "True")

The mergeSchema helps in evolving the schema of your data over time by adding new columns or modifying existing ones.<br>
To enable schema enforcement, set **mergeSchema** option to **True** when writing to a Delta table

In [0]:
data = [(1, 'foo'), (2, 'bar')]
df_mergeSchema1 = spark.createDataFrame(data, ["id", "value"])
df_mergeSchema1.show()

In [0]:
df_mergeSchema1.write.format("delta").mode("append").option("mergeSchema", "true").save('/user/deltaTables/sample')

In [0]:
# [("merged", )] - a comma is given to make python understand that we want to create a tuple with single element, else it will think it is a string
df_mergeSchema2 = spark.createDataFrame([("merged",)], ['mergeColumn'])
df_mergeSchema2.show()

In [0]:
df_mergeSchema2.write.format("delta").mode("append").option("mergeSchema", "true").save('/user/deltaTables/sample')

In [0]:
spark.read.format("delta").load('/user/deltaTables/sample').show()

### Delta Lake merge Logic

In [0]:
%sql
CREATE OR REPLACE TABLE customer_status(
id INT,
name STRING,
last_seen DATE,
status STRING)
USING DELTA;

INSERT INTO customer_status ()

In [0]:
%sql
DESCRIBE EXTENDED people

In [0]:
%sql
SELECT * FROM people

#### Create delta table from existing table

For Databricks Runtime 12 and above <br>
spark.sql("CREATE TABLE people_duplicate LIKE people")

In [0]:
data = [(1, 'Bob', 'Loblaw', 23),(2, 'Sue', 'Grafton', None), (3, 'Jim', 'Carrey', 61)]
schema = ['id', 'first_name', 'last_name', 'age']
df = spark.createDataFrame(data = data, schema = schema)

In [0]:
df.write.format("delta").saveAsTable('people')

In [0]:
spark.sql("CREATE TABLE people_duplicate AS SELECT * FROM people")

In [0]:
%sql
SELECT * FROM people_duplicate

In [0]:
%sql
DESCRIBE EXTENDED people_duplicate

### Parquet tables